# SQL Inference — Llama-3.2-3B + LoRA (vLLM)
**Model:** `meta-llama/Llama-3.2-3B-Instruct`  
**Adapter:** `glen-louis/llama-3.2-3b-sql-qlora`  

Run on Colab with GPU runtime: `Runtime → Change runtime type → T4 GPU`

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install vLLM

In [ ]:
!pip install vllm -q

## 3. Authenticate with HuggingFace

Required to download `meta-llama/Llama-3.2-3B-Instruct` (gated model).

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# vLLM V1 engine spawns a subprocess that calls sys.stdout.fileno(),
# which fails in Jupyter. Force V0 (in-process) engine instead.
os.environ["VLLM_USE_V1"] = "0"

## 4. Load Model + LoRA Adapter

In [ ]:
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
ADAPTER    = "glen-louis/llama-3.2-3b-sql-qlora"

llm = LLM(
    model=BASE_MODEL,
    enable_lora=True,
    max_lora_rank=16,
    dtype="half",          # fp16 — fits T4 VRAM
    max_model_len=2048,
)

lora_request = LoRARequest("sql-adapter", 1, ADAPTER)

print("Model loaded.")

## 5. Prompt Formatter

Matches the format used during fine-tuning (`data_utils.py`).

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def build_prompt(question: str, schema: str) -> str:
    """Matches the format used during fine-tuning (data_utils.format_prompt_only)."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a SQL expert. Given a natural language question and a database schema, "
                "write a SQL query that answers the question. Return only the SQL query with no explanation."
            ),
        },
        {
            "role": "user",
            "content": f"{question}\n\nSchema:\n{schema}",
        },
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

## 6. Run Inference

In [ ]:
def generate_sql(question: str, schema: str, max_tokens: int = 256) -> str:
    prompt = build_prompt(question, schema)
    params = SamplingParams(temperature=0, max_tokens=max_tokens)
    outputs = llm.generate([prompt], params, lora_request=lora_request)
    return outputs[0].outputs[0].text.strip()


# --- Example ---
question = "What are the top 5 customers by total order value?"

schema = """
CREATE TABLE customers (id INT, name TEXT, email TEXT);
CREATE TABLE orders (id INT, customer_id INT, amount DECIMAL, order_date DATE);
"""

sql = generate_sql(question, schema)
print(sql)

## 7. Batch Inference (optional)

Generate SQL for multiple questions at once — vLLM handles batching efficiently.

In [ ]:
examples = [
    {
        "question": "How many orders were placed in 2023?",
        "schema": "CREATE TABLE orders (id INT, customer_id INT, amount DECIMAL, order_date DATE);",
    },
    {
        "question": "List all customers who have never placed an order.",
        "schema": "CREATE TABLE customers (id INT, name TEXT); CREATE TABLE orders (id INT, customer_id INT);",
    },
    {
        "question": "What is the average salary by department?",
        "schema": "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary DECIMAL);",
    },
]

prompts = [build_prompt(ex["question"], ex["schema"]) for ex in examples]
params  = SamplingParams(temperature=0, max_tokens=256)
outputs = llm.generate(prompts, params, lora_request=lora_request)

for ex, out in zip(examples, outputs):
    print(f"Q: {ex['question']}")
    print(f"SQL: {out.outputs[0].text.strip()}")
    print("-" * 60)